# Análisis Exploratorio de Datos (EDA) - Airbnb Buenos Aires

## Objetivo
Entender la estructura, calidad y características de los datos antes de realizar transformaciones.

**Colecciones a analizar:**
- `Listings`: Información detallada de los apartamentos
- `Reviews`: Información de reseñas de clientes
- `Calendar`: Información de disponibilidad y precios

## Importaciones

In [1]:
import sys
from pathlib import Path

# Agregar el directorio padre al path para importar src
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import Extraccion, get_logger

# Configuración de visualizaciones
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

logger = get_logger(__name__)
print("✓ Importaciones completadas")

✓ Importaciones completadas


## Funciones comunes

In [2]:
from IPython.display import display

def analizar_estructura(df, nombre_coleccion):
    """
    Realiza análisis general de estructura de un DataFrame.
    
    Muestra:
    - Primeras filas
    - Dimensiones (filas y columnas)
    - Tipos de datos
    - Variables numéricas y no numéricas
    """
    print("\n" + "=" * 80)
    print(f"COLECCIÓN: {nombre_coleccion}")
    print("=" * 80)
    
    print(f"\nPRIMERAS FILAS (primeros 5 registros):")
    display(df.head(5))
    
    print(f"\nDIMENSIONES:")
    print(f"   - Filas: {df.shape[0]:,}")
    print(f"   - Columnas: {df.shape[1]}")
    
    print(f"\nTIPOS DE DATOS:")
    df.info(verbose=False)
    
    print(f"\nESTADÍSTICAS (Columnas numéricas):")
    display(df.describe().T)
    
    # Resumen de columnas no numéricas
    cols_no_numericas = df.select_dtypes(include=['object', 'str']).columns
    if len(cols_no_numericas) > 0:
        print(f"\nCOLUMNAS NO NUMÉRICAS ({len(cols_no_numericas)}):")
        for col in cols_no_numericas:
            try:
                unicos = df[col].nunique()
                print(f"   - {col}: {unicos} valores únicos")
            except TypeError:
                print(f"   - {col}: Contiene datos complejos (listas/dicts)")


def analizar_valores_nulos(df, nombre_coleccion):
    """
    Analiza valores nulos y faltantes en un DataFrame.
    
    Muestra:
    - Cantidad de nulos por columna
    - Porcentaje respecto al total de registros
    - Solo columnas que tienen al menos un nulo
    """
    print(f"\nVALORES NULOS Y FALTANTES - {nombre_coleccion}:")
    print("-" * 80)
    
    nulos = df.isnull().sum()
    total_registros = len(df)
    nulos_pct = (nulos / total_registros * 100).round(2)
    
    df_nulos = pd.DataFrame({
        'Columna': nulos.index,
        'Cantidad de nulos': nulos.values,
        'Porcentaje (%)': nulos_pct.values
    })
    
    # Filtrar solo columnas con nulos
    df_nulos_filtrado = df_nulos[df_nulos['Cantidad de nulos'] > 0].sort_values(
        'Cantidad de nulos', ascending=False
    )
    
    if len(df_nulos_filtrado) > 0:
        print(f"   Columnas con valores faltantes:")
        display(df_nulos_filtrado)
    else:
        print(f"   Todas las columnas están completas (sin nulos)")


def analizar_duplicados(df, nombre_coleccion, col_id='_id'):
    """
    Analiza registros duplicados de dos formas:
    1. Duplicados por columna ID (identidad única)
    2. Filas completamente duplicadas en todas sus columnas
    """
    print(f"\nREGISTROS DUPLICADOS - {nombre_coleccion}:")
    print("-" * 80)
    
    total = len(df)
    
    # Análisis 1: Duplicados por ID
    unicos = df[col_id].nunique()
    duplicados_id = total - unicos
    pct_duplicados_id = (duplicados_id / total * 100) if total > 0 else 0
    
    print(f"   1. POR COLUMNA ID ({col_id}):")
    print(f"      - Total de registros: {total:,}")
    print(f"      - Registros únicos: {unicos:,}")
    print(f"      - Duplicados por ID: {duplicados_id:,} ({pct_duplicados_id:.2f}%)")
    
    # Análisis 2: Filas completamente duplicadas
    filas_duplicadas = df.duplicated().sum()
    pct_filas_dup = (filas_duplicadas / total * 100) if total > 0 else 0
    
    print(f"\n   2. FILAS COMPLETAMENTE DUPLICADAS (todas las columnas):")
    print(f"      - Filas duplicadas: {filas_duplicadas:,} ({pct_filas_dup:.2f}%)")
    
    if duplicados_id == 0 and filas_duplicadas == 0:
        print(f"\n   Excelente: No hay duplicados en esta colección")


def analizar_outliers(df, nombre_coleccion, columnas=None):
    """
    Analiza outliers en columnas numéricas usando método de cuartiles (IQR).
    
    Parámetros:
    - df: DataFrame
    - nombre_coleccion: Nombre descriptivo
    - columnas: Lista de columnas específicas a analizar
              (default: detecta automáticamente todas las numéricas)
    """
    # Si no especifica columnas, detectar las numéricas automáticamente
    if columnas is None:
        columnas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    if len(columnas) == 0:
        print(f"\nANÁLISIS DE OUTLIERS - {nombre_coleccion}:")
        print("-" * 80)
        print("   No hay columnas numéricas para analizar")
        return
    
    print(f"\nANÁLISIS DE OUTLIERS - {nombre_coleccion}:")
    print("-" * 80)
    
    for col in columnas:
        if col not in df.columns:
            print(f"   Columna '{col}' no encontrada")
            continue
            
        # Filtrar valores no nulos y numéricos
        datos = pd.to_numeric(df[col], errors='coerce').dropna()
        
        if len(datos) == 0:
            print(f"\n   {col}: No hay datos numéricos válidos")
            continue
        
        Q1 = datos.quantile(0.25)
        Q3 = datos.quantile(0.75)
        IQR = Q3 - Q1
        
        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR
        
        outliers = datos[(datos < limite_inferior) | (datos > limite_superior)]
        pct_outliers = (len(outliers) / len(datos) * 100)
        
        print(f"\n   {col}:")
        print(f"      - Mínimo: {datos.min():.2f}")
        print(f"      - Q1 (25%): {Q1:.2f}")
        print(f"      - Mediana: {datos.median():.2f}")
        print(f"      - Q3 (75%): {Q3:.2f}")
        print(f"      - Máximo: {datos.max():.2f}")
        print(f"      - IQR: {IQR:.2f}")
        print(f"      - Outliers detectados: {len(outliers):,} ({pct_outliers:.2f}%)")
        print(f"      - Rango válido: [{limite_inferior:.2f}, {limite_superior:.2f}]")

## Extracción de datos desde MongoDB

In [3]:
# Conectar a MongoDB y extraer datos
extraccion = Extraccion()
print("Conectando a MongoDB...")
extraccion.conectar()

print("\nExtrayendo colecciones...")
df_listings = extraccion.extraer_coleccion("Listings")
df_reviews = extraccion.extraer_coleccion("Reviews")
df_calendar = extraccion.extraer_coleccion("Calendar")

print("\nExtracción completada")
print(f"\nRESUMEN:")
print(f"   - Listings: {len(df_listings):,} registros, {len(df_listings.columns)} columnas")
print(f"   - Reviews: {len(df_reviews):,} registros, {len(df_reviews.columns)} columnas")
print(f"   - Calendar: {len(df_calendar):,} registros, {len(df_calendar.columns)} columnas")

2026-04-11 22:17:22,096 | INFO | Extraccion | Conexion exitosa a MongoDB. Base de datos seleccionada: arbnb_ARG_BA


Conectando a MongoDB...

Extrayendo colecciones...


2026-04-11 22:17:22,932 | INFO | Extraccion | Coleccion Listings extraida correctamente. Registros encontrados: 27348. Registros cargados en DataFrame: 27348.
2026-04-11 22:17:29,857 | INFO | Extraccion | Coleccion Reviews extraida correctamente. Registros encontrados: 1042702. Registros cargados en DataFrame: 1042702.
2026-04-11 22:18:10,781 | INFO | Extraccion | Coleccion Calendar extraida correctamente. Registros encontrados: 9982072. Registros cargados en DataFrame: 9982072.



Extracción completada

RESUMEN:
   - Listings: 27,348 registros, 72 columnas
   - Reviews: 1,042,702 registros, 7 columnas
   - Calendar: 9,982,072 registros, 6 columnas


## 1. Análisis de Colección: LISTINGS

Información detallada sobre los apartamentos disponibles en Airbnb Buenos Aires.

### Análisis de estructura

In [4]:
analizar_estructura(df_listings, "LISTINGS")


COLECCIÓN: LISTINGS

PRIMERAS FILAS (primeros 5 registros):


,_id,id,listing_url,scrape_id,last_scraped,source,name,description,picture_url,host_id,...,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,reviews_per_month,host_about,license
0,69dab563f2d6c75fc1adfac8,42610838,https://www.airbnb.com/rooms/42610838,20260125052844,2026-01-25,city scrape,"Puerto Madero a 3 cuadras, centro, bello , tea...","Unbeatable location half a block away, 50 mete...",https://a0.muscache.com/pictures/miso/Hosting-...,224049389,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,69dab563f2d6c75fc1adfac9,1305876403852901802,https://www.airbnb.com/rooms/1305876403852901802,20260125052844,2026-01-25,city scrape,Apart estudio en Microcentro,"Located in the heart of Buenos Aires, this stu...",https://a0.muscache.com/pictures/miso/Hosting-...,25649070,...,3.00,4.00,2.00,3.00,3.00,5.00,4.00,0.08,NaN,NaN
2,69dab563f2d6c75fc1adfaca,1542233033640525302,https://www.airbnb.com/rooms/1542233033640525302,20260125052844,2026-01-25,city scrape,"Departamento en Buenos Aires, abasto shopping",From this central accommodation your group wil...,https://a0.muscache.com/pictures/hosting/Hosti...,153014015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Responsable amable atento y amigable,NaN
3,69dab563f2d6c75fc1adfacb,1004530078359434134,https://www.airbnb.com/rooms/1004530078359434134,20260125052844,2026-01-25,city scrape,Departamento en Recoleta,Relax with the whole family at this peaceful p...,https://a0.muscache.com/pictures/0beb83d5-381d...,1409800,...,4.73,4.64,4.64,4.82,4.73,4.59,4.45,0.80,I'm a young entrepreneur who enjoys the pleasu...,NaN
4,69dab563f2d6c75fc1adfacc,800145927121871422,https://www.airbnb.com/rooms/800145927121871422,20260125052844,2026-01-25,city scrape,Coqueto para 4 personas,3 rooms: master bedroom with king size bed and...,https://a0.muscache.com/pictures/d2a502bc-2ac0...,467965425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"amante de la naturaleza, aire libre y aliment...",NaN



DIMENSIONES:
   - Filas: 27,348
   - Columnas: 72

TIPOS DE DATOS:
<class 'pandas.DataFrame'>
RangeIndex: 27348 entries, 0 to 27347
Columns: 72 entries, _id to license
dtypes: bool(3), datetime64[us](4), float64(21), int64(24), object(3), str(17)
memory usage: 14.5+ MB

ESTADÍSTICAS (Columnas numéricas):


,count,mean,min,25%,50%,75%,max,std
id,27348.0,855677951028418560.0,11508.0,562603542623496320.0,988672357895861376.0,1293344149331353600.0,1605544802375593472.0,547796918634951744.0
scrape_id,27348.0,20260125052844.0,20260125052844.0,20260125052844.0,20260125052844.0,20260125052844.0,20260125052844.0,0.0
last_scraped,27348,2026-01-25 11:36:53.075910,2026-01-25 00:00:00,2026-01-25 00:00:00,2026-01-25 00:00:00,2026-01-26 00:00:00,2026-01-26 00:00:00,NaN
host_id,27348.0,237276277.529179,13426.0,34273413.25,158701702.5,460093620.5,742015086.0,223350508.008803
host_profile_id,27348.0,1469891665664825088.0,1462506732948986624.0,1462898567844714240.0,1466928187478477056.0,1469954363744958976.0,1605075436443496448.0,18375673888162604.0
hosts_time_as_user_years,27348.0,7.263346,0.0,3.0,8.0,10.0,16.0,3.984068
hosts_time_as_user_months,27348.0,5.282105,0.0,2.0,5.0,8.0,11.0,3.53407
hosts_time_as_host_years,27348.0,5.135695,0.0,2.0,4.0,8.0,15.0,3.966324
hosts_time_as_host_months,27348.0,5.067756,0.0,2.0,5.0,8.0,11.0,3.546789
host_listings_count,27348.0,17.92398,1.0,1.0,3.0,16.0,945.0,38.39527



COLUMNAS NO NUMÉRICAS (20):
   - _id: 27348 valores únicos
   - listing_url: 27348 valores únicos
   - source: 1 valores únicos
   - name: 25865 valores únicos
   - description: 23703 valores únicos
   - picture_url: 26848 valores únicos
   - host_url: 14355 valores únicos
   - host_profile_url: 13719 valores únicos
   - host_name: 3841 valores únicos
   - host_picture_url: 12960 valores únicos
   - host_verifications: 1 valores únicos
   - neighbourhood_cleansed: 48 valores únicos
   - property_type: 58 valores únicos
   - room_type: 4 valores únicos
   - bathrooms_text: 47 valores únicos
   - amenities: Contiene datos complejos (listas/dicts)
   - has_availability: 1 valores únicos
   - host_location: 650 valores únicos
   - host_about: 6053 valores únicos
   - license: 423 valores únicos


### Análisis de valores nulos

In [5]:
analizar_valores_nulos(df_listings, "LISTINGS")


VALORES NULOS Y FALTANTES - LISTINGS:
--------------------------------------------------------------------------------
   Columnas con valores faltantes:


,Columna,Cantidad de nulos,Porcentaje (%)
71,license,26886,98.31
70,host_about,11783,43.09
59,host_location,5783,21.15
63,review_scores_accuracy,3304,12.08
64,review_scores_cleanliness,3303,12.08
62,review_scores_rating,3303,12.08
61,last_review,3303,12.08
60,first_review,3303,12.08
66,review_scores_communication,3303,12.08
67,review_scores_location,3303,12.08


### Análisis de duplicados

In [6]:
analizar_duplicados(df_listings, "LISTINGS", col_id='id')


REGISTROS DUPLICADOS - LISTINGS:
--------------------------------------------------------------------------------
   1. POR COLUMNA ID (id):
      - Total de registros: 27,348
      - Registros únicos: 27,348
      - Duplicados por ID: 0 (0.00%)


TypeError: unhashable type: 'list'

### Análisis de outliers

In [ ]:
analizar_outliers(df_listings, "LISTINGS")

## 2. Análisis de Colección: REVIEWS

Información sobre las reseñas que dejan los usuarios de los apartamentos.

### Análisis de estructura

In [ ]:
analizar_estructura(df_reviews, "REVIEWS")

### Análisis de valores nulos

In [ ]:
analizar_valores_nulos(df_reviews, "REVIEWS")

### Análisis de duplicados

In [ ]:
analizar_duplicados(df_reviews, "REVIEWS", col_id='_id')

### Análisis de outliers

In [ ]:
analizar_outliers(df_reviews, "REVIEWS")

## 3. Análisis de Colección: CALENDAR

Información sobre disponibilidad, precios y políticas de noches mínimas para cada día.

### Análisis de estructura

In [ ]:
analizar_estructura(df_calendar, "CALENDAR")

### Análisis de valores nulos

In [ ]:
analizar_valores_nulos(df_calendar, "CALENDAR")

### Análisis de duplicados

In [ ]:
analizar_duplicados(df_calendar, "CALENDAR", col_id='_id')

### Análisis de outliers

In [ ]:
analizar_outliers(df_calendar, "CALENDAR")